# Exploración de datos y regresión lineal paso a paso

Este laboratorio usa el dataset de solubilidad de Delaney para contestar preguntas antes de modelar:

1. ¿Cómo luce cada columna (mínimo, máximo, promedio, dispersión y valores atípicos)?
2. ¿Qué descriptor tiene la relación lineal más fuerte con `logS`?
3. ¿Cómo calcula una recta sus parámetros y sus errores?
4. ¿Predice mejor una sola variable o todas las variables?
5. ¿Cuándo funciona bien una regresión lineal y cuándo falla?


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")
plt.style.use("seaborn-v0_8-whitegrid")

URL = "https://raw.githubusercontent.com/dataprofessor/data/master/delaney_solubility_with_descriptors.csv"
TARGET = "logS"
df = pd.read_csv(URL)
features = [column for column in df.columns if column != TARGET]

print(f"Filas: {len(df):,} | Columnas: {df.shape[1]}")
display(df.head())

## 1. Auditoría inicial: ¿podemos confiar en la tabla?

Antes de buscar patrones comprobamos tipos, valores ausentes y duplicados. Una regresión necesita números y no acepta valores ausentes sin un tratamiento previo.

- **Característica (`X`)**: dato disponible para explicar o predecir.
- **Objetivo (`y`)**: resultado que queremos predecir; aquí es `logS`.
- Cada fila representa una molécula; cada columna, una medición de esa molécula.

In [ ]:
auditoria = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "faltantes": df.isna().sum(),
    "% faltantes": df.isna().mean().mul(100),
    "valores únicos": df.nunique(),
})

display(auditoria)
print(f"Filas duplicadas: {df.duplicated().sum()}")
print("Todas las columnas son numéricas:", all(pd.api.types.is_numeric_dtype(df[c]) for c in df.columns))

## 2. Perfil de cada columna

Las medidas responden preguntas diferentes:

- **min / max**: rango observado.
- **media**: centro aritmético, sensible a extremos.
- **mediana (50%)**: centro resistente a extremos.
- **desviación estándar**: dispersión alrededor de la media.
- **Q1 (25%) y Q3 (75%)**: contienen la mitad central de los datos.
- **IQR = Q3 − Q1**: dispersión robusta; ayuda a identificar valores atípicos.
- **asimetría**: cerca de 0 es más simétrica; magnitud alta indica una cola larga.

Estas escalas no son comparables directamente. Un coeficiente grande no implica por sí solo que una variable sea más importante.

In [ ]:
perfil = df.describe().T.rename(columns={
    "count": "n", "mean": "media", "std": "desv_est",
    "min": "mínimo", "25%": "Q1", "50%": "mediana",
    "75%": "Q3", "max": "máximo",
})
perfil["IQR"] = perfil["Q3"] - perfil["Q1"]
perfil["asimetría"] = df.skew()
perfil["atípicos_IQR"] = [
    ((df[col] < perfil.loc[col, "Q1"] - 1.5 * perfil.loc[col, "IQR"]) |
     (df[col] > perfil.loc[col, "Q3"] + 1.5 * perfil.loc[col, "IQR"])).sum()
    for col in perfil.index
]
display(perfil)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, column in zip(axes.flat, df.columns):
    ax.hist(df[column], bins=30, edgecolor="white")
    ax.axvline(df[column].mean(), color="crimson", linestyle="--", label="media")
    ax.axvline(df[column].median(), color="black", linestyle=":", label="mediana")
    ax.set_title(column)
    ax.legend()
axes.flat[-1].axis("off")
fig.suptitle("Distribuciones: forma, centro y dispersión", fontsize=14)
plt.tight_layout()
plt.show()

## 3. Explorar con filtros y grupos

Un resumen global puede ocultar segmentos. Los filtros permiten preguntar, por ejemplo: “¿cómo cambia `logS` en moléculas con `MolLogP` bajo, medio o alto?”.

Los cortes siguientes son **cuartiles**, por lo que cada grupo contiene aproximadamente 25% de las observaciones. Esto facilita una comparación equilibrada; no establece causalidad.

In [ ]:
q1, q3 = df["MolLogP"].quantile([0.25, 0.75])

filtros = {
    "MolLogP bajo (<= Q1)": df[df["MolLogP"] <= q1],
    "MolLogP medio (Q1 a Q3)": df[df["MolLogP"].between(q1, q3)],
    "MolLogP alto (>= Q3)": df[df["MolLogP"] >= q3],
}

resumen_filtros = pd.DataFrame([
    {
        "grupo": nombre,
        "n": len(datos),
        "MolLogP medio": datos["MolLogP"].mean(),
        "logS medio": datos[TARGET].mean(),
        "logS mínimo": datos[TARGET].min(),
        "logS máximo": datos[TARGET].max(),
    }
    for nombre, datos in filtros.items()
]).set_index("grupo")

display(resumen_filtros)

# Modifica estos límites para formular tus propias preguntas.
molwt_mediana = df["MolWt"].median()
ejemplo_filtrado = df.query("MolLogP >= @q3 and MolWt >= @molwt_mediana")
print("Ejemplo: MolLogP alto y peso molecular sobre la mediana")
display(ejemplo_filtrado.sort_values(TARGET).head(10))

## 4. Correlación con el objetivo

La correlación de Pearson mide la **fuerza y dirección de una relación lineal** entre −1 y 1:

- cerca de `+1`: ambas variables tienden a crecer juntas;
- cerca de `−1`: una tiende a bajar cuando la otra sube;
- cerca de `0`: no hay una relación **lineal** clara.

Correlación no implica causalidad y puede ser engañosa ante valores atípicos o curvas. Por eso combinamos la tabla con diagramas de dispersión.

In [ ]:
correlaciones = (
    df.corr(numeric_only=True)[TARGET]
      .drop(TARGET)
      .rename("correlación con logS")
      .to_frame()
)
correlaciones["correlación absoluta"] = correlaciones["correlación con logS"].abs()
correlaciones = correlaciones.sort_values("correlación absoluta", ascending=False)
most_significant = correlaciones.index[0]

display(correlaciones)
print(f"Mayor relación lineal absoluta: {most_significant} ({correlaciones.iloc[0, 0]:.3f})")

corr_matrix = df.corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(7, 6))
image = ax.imshow(corr_matrix, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_matrix)), corr_matrix.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr_matrix)), corr_matrix.index)
for i in range(len(corr_matrix)):
    for j in range(len(corr_matrix)):
        ax.text(j, i, f"{corr_matrix.iloc[i, j]:.2f}", ha="center", va="center")
fig.colorbar(image, ax=ax, label="correlación")
ax.set_title("Matriz de correlación")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, feature in zip(axes.flat, features):
    ax.scatter(df[feature], df[TARGET], alpha=0.35, s=18)
    slope, intercept = np.polyfit(df[feature], df[TARGET], 1)
    x_line = np.linspace(df[feature].min(), df[feature].max(), 100)
    ax.plot(x_line, intercept + slope * x_line, color="crimson", linewidth=2)
    ax.set(xlabel=feature, ylabel=TARGET,
           title=f"r = {df[[feature, TARGET]].corr().iloc[0, 1]:.3f}")
fig.suptitle("Cada característica frente al objetivo", fontsize=14)
plt.tight_layout()
plt.show()

## 5. La recta por dentro: un subconjunto diminuto

Para una sola variable, la regresión busca la recta

$$\hat y = b_0 + b_1x$$

que minimiza la **suma de errores cuadrados** $\sum(y-\hat y)^2$.

$$b_1=\frac{\sum(x_i-\bar x)(y_i-\bar y)}{\sum(x_i-\bar x)^2}, \qquad b_0=\bar y-b_1\bar x$$

- $b_1$ (pendiente): cambio esperado en `logS` cuando `x` aumenta una unidad.
- $b_0$ (intercepto): predicción cuando `x=0`; puede no tener interpretación física.
- residuo $e=y-\hat y$: distancia vertical entre el dato y la recta.

Tomamos 8 filas distribuidas a lo largo del rango de la característica más correlacionada. Es una demostración visual, **no** una muestra para evaluar rendimiento.

In [ ]:
ordenados = df.sort_values(most_significant).reset_index(drop=True)
posiciones = np.linspace(0, len(ordenados) - 1, 8, dtype=int)
tiny = ordenados.iloc[posiciones][[most_significant, TARGET]].copy()

x = tiny[most_significant].to_numpy()
y = tiny[TARGET].to_numpy()
x_mean, y_mean = x.mean(), y.mean()

numerador = ((x - x_mean) * (y - y_mean)).sum()
denominador = ((x - x_mean) ** 2).sum()
b1_manual = numerador / denominador
b0_manual = y_mean - b1_manual * x_mean

tiny["x - media(x)"] = x - x_mean
tiny["y - media(y)"] = y - y_mean
tiny["producto_desvíos"] = tiny["x - media(x)"] * tiny["y - media(y)"]
tiny["predicción"] = b0_manual + b1_manual * x
tiny["residuo"] = y - tiny["predicción"]
tiny["residuo²"] = tiny["residuo"] ** 2

display(tiny)
print(f"Numerador = {numerador:.3f}; denominador = {denominador:.3f}")
print(f"Pendiente b1 = {b1_manual:.3f}; intercepto b0 = {b0_manual:.3f}")
print(f"Recta manual: logS = {b0_manual:.3f} + ({b1_manual:.3f} × {most_significant})")
print(f"Suma de errores cuadrados = {tiny['residuo²'].sum():.3f}")

modelo_tiny = LinearRegression().fit(tiny[[most_significant]], y)
print("Coincide con sklearn:",
      np.isclose(modelo_tiny.coef_[0], b1_manual) and
      np.isclose(modelo_tiny.intercept_, b0_manual))

x_line = np.linspace(x.min(), x.max(), 100)
plt.figure(figsize=(9, 5))
plt.scatter(x, y, s=65, label="datos observados")
plt.plot(x_line, b0_manual + b1_manual * x_line, color="crimson", label="recta ajustada")
for xi, yi, y_hat in zip(x, y, tiny["predicción"]):
    plt.plot([xi, xi], [yi, y_hat], color="gray", linestyle="--", alpha=0.8)
plt.xlabel(most_significant)
plt.ylabel(TARGET)
plt.title("La recta minimiza las distancias verticales al cuadrado")
plt.legend()
plt.show()

## 6. Comparación justa: promedio vs. una variable vs. todas

Ahora separamos datos de **entrenamiento** (la recta aprende) y **prueba** (evaluación con moléculas no vistas). Los tres modelos usan exactamente las mismas filas de prueba.

- **Línea base**: siempre predice el promedio de `logS` del entrenamiento.
- **Regresión simple**: usa solo la característica con mayor correlación absoluta.
- **Regresión múltiple**: usa las cuatro características.

Métricas:
- **MAE**: error absoluto medio, en unidades de `logS`; menor es mejor.
- **RMSE**: penaliza más los errores grandes; menor es mejor.
- **R²**: fracción de variación explicada frente a predecir el promedio; mayor es mejor. Puede ser negativo en prueba.

Agregar columnas **no garantiza** mejorar en datos nuevos: una variable irrelevante puede introducir ruido. La prueba empírica decide.

In [ ]:
train_idx, test_idx = train_test_split(
    df.index, test_size=0.20, random_state=100
)
y_train = df.loc[train_idx, TARGET]
y_test = df.loc[test_idx, TARGET]

model_simple = LinearRegression().fit(
    df.loc[train_idx, [most_significant]], y_train
)
model_multiple = LinearRegression().fit(
    df.loc[train_idx, features], y_train
)

predictions = {
    "Promedio (línea base)": np.repeat(y_train.mean(), len(test_idx)),
    f"Simple: {most_significant}": model_simple.predict(df.loc[test_idx, [most_significant]]),
    "Múltiple: todas": model_multiple.predict(df.loc[test_idx, features]),
}

def metricas(nombre, y_real, y_pred):
    return {
        "modelo": nombre,
        "MAE": mean_absolute_error(y_real, y_pred),
        "RMSE": mean_squared_error(y_real, y_pred) ** 0.5,
        "R²": r2_score(y_real, y_pred),
    }

comparacion = pd.DataFrame([
    metricas(nombre, y_test, pred)
    for nombre, pred in predictions.items()
]).set_index("modelo")
display(comparacion)

r2_simple = comparacion.loc[f"Simple: {most_significant}", "R²"]
r2_multiple = comparacion.loc["Múltiple: todas", "R²"]
print(f"La variable {most_significant} explica {r2_simple:.1%} de la variación de prueba.")
print(f"Todas las variables explican {r2_multiple:.1%}.")
print(f"Cambio al agregar variables: {r2_multiple - r2_simple:+.3f} puntos de R².")
print(f"Ecuación simple: logS = {model_simple.intercept_:.3f} + "
      f"({model_simple.coef_[0]:.3f} × {most_significant})")

comparacion[["MAE", "RMSE"]].plot.bar(figsize=(9, 5), rot=0)
plt.title("Error en datos no vistos (más bajo es mejor)")
plt.ylabel("error")
plt.show()

## 7. ¿Cómo combina información la regresión múltiple?

La recta se convierte en un plano de varias dimensiones:

$$\hat y=b_0+b_1x_1+b_2x_2+\cdots+b_px_p$$

Cada coeficiente representa el cambio esperado en `logS` por una unidad de esa columna **manteniendo las demás constantes**. Como las unidades difieren (`MolWt` puede medir cientos y `AromaticProportion` entre 0 y 1), también ajustamos un modelo estandarizado. Allí, el valor absoluto permite comparar cuánto cambia la predicción al mover cada variable una desviación estándar.

Esto describe asociación predictiva, no causalidad.

In [ ]:
coeficientes = pd.DataFrame({
    "coeficiente original": model_multiple.coef_
}, index=features)

standard_model = make_pipeline(StandardScaler(), LinearRegression())
standard_model.fit(df.loc[train_idx, features], y_train)
coeficientes["coeficiente estandarizado"] = standard_model[-1].coef_
coeficientes["magnitud estandarizada"] = coeficientes["coeficiente estandarizado"].abs()
coeficientes = coeficientes.sort_values("magnitud estandarizada", ascending=False)

display(coeficientes)
print(f"Intercepto en unidades originales: {model_multiple.intercept_:.3f}")

coeficientes["coeficiente estandarizado"].sort_values().plot.barh(
    figsize=(8, 4), color=["crimson" if value < 0 else "steelblue"
                           for value in coeficientes["coeficiente estandarizado"].sort_values()]
)
plt.axvline(0, color="black", linewidth=1)
plt.title("Efecto parcial comparable (variables estandarizadas)")
plt.xlabel("cambio en logS por +1 desviación estándar")
plt.show()

## 8. ¿Dónde funciona y dónde falla?

Una regresión lineal es apropiada cuando la relación es aproximadamente lineal, los errores no forman patrones claros y necesitamos una explicación simple. Puede fallar con curvas, interacciones complejas, valores atípicos, extrapolación o variables importantes ausentes.

Abajo inspeccionamos el modelo múltiple de tres maneras:

1. **Real vs. predicho**: idealmente los puntos siguen la diagonal.
2. **Residuos vs. predicción**: idealmente forman una nube aleatoria alrededor de cero.
3. **Tabla de mejores y peores casos**: revela moléculas que la ecuación representa bien o mal.

Un residuo grande no significa que la fila sea incorrecta; puede indicar información química que estas cuatro columnas no capturan.

In [ ]:
diagnostico = df.loc[test_idx, features + [TARGET]].copy()
diagnostico["predicción"] = predictions["Múltiple: todas"]
diagnostico["residuo"] = diagnostico[TARGET] - diagnostico["predicción"]
diagnostico["error absoluto"] = diagnostico["residuo"].abs()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(diagnostico[TARGET], diagnostico["predicción"], alpha=0.55)
limites = [min(diagnostico[TARGET].min(), diagnostico["predicción"].min()),
           max(diagnostico[TARGET].max(), diagnostico["predicción"].max())]
axes[0].plot(limites, limites, "r--", label="predicción perfecta")
axes[0].set(xlabel="logS real", ylabel="logS predicho", title="Real vs. predicho")
axes[0].legend()

axes[1].scatter(diagnostico["predicción"], diagnostico["residuo"], alpha=0.55)
axes[1].axhline(0, color="crimson", linestyle="--")
axes[1].set(xlabel="logS predicho", ylabel="residuo (real − predicho)",
            title="¿Queda algún patrón sin aprender?")
plt.tight_layout()
plt.show()

columnas_tabla = features + [TARGET, "predicción", "residuo", "error absoluto"]
print("5 casos mejor representados por el plano lineal")
display(diagnostico.nsmallest(5, "error absoluto")[columnas_tabla])
print("5 casos peor representados por el plano lineal")
display(diagnostico.nlargest(5, "error absoluto")[columnas_tabla])

# Agrupar la predicción permite detectar sesgo sistemático por zona.
diagnostico["zona predicha"] = pd.qcut(diagnostico["predicción"], 5, duplicates="drop")
por_zona = diagnostico.groupby("zona predicha", observed=True).agg(
    n=(TARGET, "size"),
    logS_real_medio=(TARGET, "mean"),
    logS_predicho_medio=("predicción", "mean"),
    residuo_medio=("residuo", "mean"),
    MAE=("error absoluto", "mean"),
)
display(por_zona)

## 9. Tabla de decisión: ¿es ideal usar regresión lineal?

| Evidencia observada | Regresión lineal suele ser adecuada | Señal de que puede no bastar |
|---|---|---|
| Dispersión `X`–`y` | nube alrededor de una recta | curva, escalones o grupos separados |
| Residuos | aleatorios alrededor de 0 | forma de U, embudo o grupos |
| Valores atípicos | pocos y justificables | puntos extremos dominan la pendiente |
| R² de prueba | estable y útil para el objetivo | mucho menor que entrenamiento o negativo |
| Necesidad de explicación | coeficientes claros son importantes | precisión domina y hay patrones complejos |
| Extrapolación | rara y cercana al rango observado | predicciones lejos del rango de entrenamiento |

### Lo que este laboratorio demuestra

- Explorar distribución, calidad, filtros y correlaciones precede al modelado.
- La correlación ayuda a escoger una primera variable, pero no reemplaza una evaluación en prueba.
- La regresión aprende coeficientes minimizando residuos cuadrados en entrenamiento.
- Agregar variables puede mejorar el contexto predictivo, pero se acepta solo si mejora datos no vistos.
- Tablas de errores y gráficos de residuos muestran *dónde* el promedio global de métricas oculta fallos.

### Experimentos sugeridos

1. Cambia `random_state` y observa cuánto varía R².
2. Escoge manualmente otra variable y repite la regresión simple.
3. Elimina los valores atípicos usando IQR, vuelve a entrenar y explica el riesgo de hacerlo.
4. Agrega características una por una y registra el R² de prueba.
5. Compara posteriormente con Random Forest en el notebook principal; pregunta qué patrones no lineales logra capturar.